① Flat 결과 vs Hier 결과를 같은 Gold로 평가 (Top-1 / Top-3)  

In [3]:
import pandas as pd
from math import comb

# =========================
# 경로 설정
# =========================
GOLD_PATH = "datas/processed/gold_final_20.csv"
FLAT_PATH = "outputs/sbert_pilot_top3.csv"
HIER_PATH = "outputs/sbert_hier_top3.csv"   # Hierarchical Alignment 결과 파일

# =========================
# 유틸: Top-1 / Top-3 평가 함수
# =========================
def prepare_gold(gold_path: str):
    gold = pd.read_csv(gold_path, dtype={"job_posting_id": "string"})
    if "job_posting_id" not in gold.columns or "gold_final" not in gold.columns:
        raise ValueError("Gold 파일에 'job_posting_id', 'gold_final' 컬럼이 필요합니다.")
    gold["job_posting_id"] = gold["job_posting_id"].astype(str).str.strip()
    gold["gold_final"] = gold["gold_final"].astype(str).str.strip()
    return gold[["job_posting_id", "gold_final"]].drop_duplicates("job_posting_id")

def prepare_pred_topk(pred_path: str, label_col="job_role_name", topk=3):
    pred = pd.read_csv(pred_path, dtype={"job_posting_id": "string"})
    need_cols = {"job_posting_id", "rank", label_col}
    if not need_cols.issubset(set(pred.columns)):
        raise ValueError(f"예측 파일에 필수 컬럼이 없습니다. 필요={need_cols}, 현재={set(pred.columns)}")

    pred["job_posting_id"] = pred["job_posting_id"].astype(str).str.strip()
    pred["rank"] = pred["rank"].astype(str).str.strip()
    pred[label_col] = pred[label_col].astype(str).str.strip()

    # rank가 '1','2','3' 또는 '1.0' 형태일 수 있으니 처리
    valid_ranks = {str(i) for i in range(1, topk+1)} | {f"{i}.0" for i in range(1, topk+1)}
    pred_k = pred[pred["rank"].isin(valid_ranks)].copy()

    # Top-1
    top1 = pred_k[pred_k["rank"].isin(["1","1.0"])][["job_posting_id", label_col]].rename(
        columns={label_col: "pred_top1"}
    )

    # Top-k 리스트
    topk_list = (pred_k.groupby("job_posting_id")[label_col]
                 .apply(list)
                 .reset_index()
                 .rename(columns={label_col: "pred_topk_list"}))

    return top1, topk_list

def eval_top1_topk(gold_df, top1_df, topk_df):
    # Top-1
    e1 = gold_df.merge(top1_df, on="job_posting_id", how="inner")
    e1["correct_top1"] = (e1["gold_final"] == e1["pred_top1"])
    top1_acc = e1["correct_top1"].mean() if len(e1) else float("nan")

    # Top-k
    ek = gold_df.merge(topk_df, on="job_posting_id", how="inner")
    ek["hit_topk"] = ek.apply(lambda r: r["gold_final"] in r["pred_topk_list"], axis=1)
    topk_acc = ek["hit_topk"].mean() if len(ek) else float("nan")

    return e1, ek, top1_acc, topk_acc

# =========================
# 1) Gold / Flat / Hier 로드 및 평가
# =========================
gold = prepare_gold(GOLD_PATH)

flat_top1, flat_top3 = prepare_pred_topk(FLAT_PATH, label_col="job_role_name", topk=3)
hier_top1, hier_top3 = prepare_pred_topk(HIER_PATH, label_col="job_role_name", topk=3)

flat_e1, flat_e3, flat_top1_acc, flat_top3_acc = eval_top1_topk(gold, flat_top1, flat_top3)
hier_e1, hier_e3, hier_top1_acc, hier_top3_acc = eval_top1_topk(gold, hier_top1, hier_top3)

print("========== (1) Top-1 / Top-3 평가 ==========")
print(f"[Flat] Top-1 Accuracy: {flat_top1_acc:.4f}  ({flat_e1['correct_top1'].sum()}/{len(flat_e1)})")
print(f"[Flat] Top-3 Accuracy: {flat_top3_acc:.4f}  ({flat_e3['hit_topk'].sum()}/{len(flat_e3)})")
print()
print(f"[Hier] Top-1 Accuracy: {hier_top1_acc:.4f}  ({hier_e1['correct_top1'].sum()}/{len(hier_e1)})")
print(f"[Hier] Top-3 Accuracy: {hier_top3_acc:.4f}  ({hier_e3['hit_topk'].sum()}/{len(hier_e3)})")



========== (1) Top-1 / Top-3 평가 ==========
[Flat] Top-1 Accuracy: 0.1500  (3/20)
[Flat] Top-3 Accuracy: 0.5500  (11/20)

[Hier] Top-1 Accuracy: 0.1500  (3/20)
[Hier] Top-3 Accuracy: 0.5500  (11/20)


[해석]  
Top-1/Top-3  
Hier가 Flat보다 Top-1 또는 Top-3에서 높으면 → “계층 정렬이 유리”한 신호

② Flat vs Hier McNemar test (Top-1 기준)

In [4]:
# =========================
# 2) McNemar test (Top-1 기준)
#    - 같은 샘플(job_posting_id)에서 Flat/Hier의 정오표 비교
#    - 2x2 표:
#        a = 둘다 정답
#        b = Flat 정답, Hier 오답
#        c = Flat 오답, Hier 정답
#        d = 둘다 오답
# =========================
print("\n========== (2) McNemar test (Top-1) ==========")

# Flat/Hier Top-1 정오 결합
cmp = gold.merge(flat_e1[["job_posting_id","correct_top1"]].rename(columns={"correct_top1":"flat_correct"}),
                 on="job_posting_id", how="inner")
cmp = cmp.merge(hier_e1[["job_posting_id","correct_top1"]].rename(columns={"correct_top1":"hier_correct"}),
                on="job_posting_id", how="inner")

if len(cmp) == 0:
    raise ValueError("McNemar: 비교 가능한 공고가 없습니다. (ID 매칭 확인)")

a = int(((cmp["flat_correct"] == True)  & (cmp["hier_correct"] == True)).sum())
b = int(((cmp["flat_correct"] == True)  & (cmp["hier_correct"] == False)).sum())
c = int(((cmp["flat_correct"] == False) & (cmp["hier_correct"] == True)).sum())
d = int(((cmp["flat_correct"] == False) & (cmp["hier_correct"] == False)).sum())

print(f"Contingency table (Top-1):")
print(f"  a(both correct) = {a}")
print(f"  b(flat correct, hier wrong) = {b}")
print(f"  c(flat wrong, hier correct) = {c}")
print(f"  d(both wrong) = {d}")
print(f"  N = {a+b+c+d}")

# --- McNemar exact p-value (권장: 표본 작을 때) ---
# p = 2 * P(X <= min(b,c)) where X ~ Binomial(n=b+c, p=0.5)
def mcnemar_exact_p(b, c):
    n = b + c
    if n == 0:
        return 1.0
    k = min(b, c)
    # cumulative probability
    p = 0.0
    for i in range(0, k+1):
        p += comb(n, i) * (0.5 ** n)
    p = 2 * p
    return min(1.0, p)

p_exact = mcnemar_exact_p(b, c)

# --- McNemar chi-square (연속성 보정) ---
# chi2 = (|b-c|-1)^2 / (b+c)
if (b + c) == 0:
    chi2_cc = 0.0
else:
    chi2_cc = ((abs(b - c) - 1) ** 2) / (b + c)

print(f"\nMcNemar exact p-value (two-sided): {p_exact:.6f}")
print(f"McNemar chi-square (cc): {chi2_cc:.4f}  (df=1)")
print("Decision (alpha=0.05):", "SIGNIFICANT" if p_exact < 0.05 else "NOT significant")


========== (2) McNemar test (Top-1) ==========
Contingency table (Top-1):
  a(both correct) = 3
  b(flat correct, hier wrong) = 0
  c(flat wrong, hier correct) = 0
  d(both wrong) = 17
  N = 20

McNemar exact p-value (two-sided): 1.000000
McNemar chi-square (cc): 0.0000  (df=1)
Decision (alpha=0.05): NOT significant


[해석]

McNemar (Top-1)  
p < 0.05면 → Flat vs Hier 차이가 우연이 아니라 “유의미” (KIIT에서 좋아하는 결론)